# Adaptive Dual-Branch HAT–CNN Fusion

## الفكرة

نستخدم أفضل نموذجين تم تدريبهما سابقًا كخبيرين مستقلين:

```text
Expert 1: WV3-Pretrained HAT-PAN
Expert 2: Fusion CNN
```

كل خبير يرى المدخلات الأصلية:

```text
LR-MS: 6 × 64 × 64
PAN:   1 × 256 × 256
```

ثم ندمج Residuals الخاصة بهما بواسطة بوابة تكيفية:

```text
Bicubic MS
    +
w_HAT(x,y,b) × HAT Residual
    +
w_CNN(x,y,b) × CNN Residual
    ↓
Final 6-band HR-MS
```

الأوزان:

```text
w_HAT + w_CNN = 1
```

وتختلف حسب:

- البكسل.
- الـBand.
- نوع المنطقة.
- مقدار اختلاف HAT عن CNN.
- التفاصيل الموجودة في PAN.

## Loss الجديدة الخاصة بالدمج

### 1. Charbonnier Reconstruction Loss

نسخة مستقرة من L1 لإعادة بناء Target.

### 2. Spectral Angle Loss

تحافظ على العلاقة الطيفية بين الـ6 Bands.

### 3. Gradient Loss

تحافظ على الحواف والتفاصيل المكانية.

### 4. PAN High-Pass Correlation Loss

تشجع تفاصيل الناتج على التوافق مع تفاصيل PAN، دون فرض أن قيم PAN تساوي قيم الـMS.

### 5. Low-Resolution Spectral Consistency

عند تصغير الناتج ×4 يجب أن يعود قريبًا من LR-MS الأصلية.

### 6. Oracle-Guided Gate Loss — الإضافة الأساسية

نحسب أثناء التدريب أي فرع أقرب إلى Target عند كل Pixel وكل Band:

```text
HAT error مقابل CNN error
```

ثم نعلّم البوابة أن تعطي وزنًا أكبر للفرع الأفضل في هذا الموضع.

### 7. Branch Stabilization Loss

أثناء Fine-tuning تمنع فرعي HAT وCNN من فقد الأوزان الجيدة التي تعلماها سابقًا.


## 1) تفعيل GPU

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "فعّل T4 GPU من Runtime → Change runtime type."
    )

print("GPU:", torch.cuda.get_device_name(0))

torch.set_float32_matmul_precision("high")


PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


## 2) تنزيل HAT وتثبيت المكتبات

لا نعمل Upgrade لـNumPy.


In [2]:
from pathlib import Path
import shutil
import subprocess
import importlib.util

%cd /content

HAT_REPO = Path("/content/HAT")

if HAT_REPO.exists():
    shutil.rmtree(HAT_REPO)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/XPixelGroup/HAT.git",
        str(HAT_REPO),
    ],
    check=True,
)

!pip install -q einops timm matplotlib tqdm

print("Repository:", HAT_REPO)


/content
Repository: /content/HAT


## 3) استيراد HAT دون تحميل BasicSR بالكامل

In [3]:
ORIGINAL_ARCH = (
    HAT_REPO
    / "hat"
    / "archs"
    / "hat_arch.py"
)

STANDALONE_ARCH = Path(
    "/content/hat_arch_standalone.py"
)

source = ORIGINAL_ARCH.read_text(
    encoding="utf-8"
)

source = source.replace(
    "from basicsr.utils.registry import ARCH_REGISTRY",
    """
class _SimpleRegistry:
    def register(self):
        def decorator(obj):
            return obj
        return decorator

ARCH_REGISTRY = _SimpleRegistry()
""".strip(),
)

source = source.replace(
    "from basicsr.archs.arch_util import to_2tuple, trunc_normal_",
    "from timm.layers import to_2tuple, trunc_normal_",
)

STANDALONE_ARCH.write_text(
    source,
    encoding="utf-8",
)

spec = importlib.util.spec_from_file_location(
    "hat_arch_standalone",
    STANDALONE_ARCH,
)

hat_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hat_module)

HAT = hat_module.HAT

print("HAT imported successfully.")


HAT imported successfully.


## 4) ربط Google Drive والمسارات

In [4]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Super_Resolution_28-07-2026"
)

PATCHES_DIR = (
    PROJECT_DIR
    / "Wald_Data_GSD"
    / "Patches"
)

TRAIN_DIR = PATCHES_DIR / "train"
VAL_DIR = PATCHES_DIR / "val"
TEST_DIR = PATCHES_DIR / "test"

HAT_MODEL_PATH = (
    PROJECT_DIR
    / "WV3_Transfer_HAT_Results"
    / "best_wv3_transfer_hat_6band.pth"
)

CNN_MODEL_PATH = (
    PROJECT_DIR
    / "Fusion_Baseline_Results"
    / "best_fusion_baseline.pth"
)

NORMALIZATION_PATH = (
    PROJECT_DIR
    / "Fusion_Baseline_Results"
    / "train_normalization_stats.json"
)

CNN_METRICS_PATH = (
    PROJECT_DIR
    / "Fusion_Baseline_Results"
    / "test_metrics.json"
)

HAT_METRICS_PATH = (
    PROJECT_DIR
    / "WV3_Transfer_HAT_Results"
    / "transfer_test_metrics.json"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "Adaptive_HAT_CNN_Fusion_Results"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

STAGE_A_PATH = (
    OUTPUT_DIR
    / "best_gate_stage_a.pth"
)

BEST_MODEL_PATH = (
    OUTPUT_DIR
    / "best_adaptive_hat_cnn_fusion.pth"
)

LAST_MODEL_PATH = (
    OUTPUT_DIR
    / "last_adaptive_hat_cnn_fusion.pth"
)

HISTORY_PATH = (
    OUTPUT_DIR
    / "adaptive_fusion_history.json"
)

TEST_METRICS_PATH = (
    OUTPUT_DIR
    / "adaptive_fusion_test_metrics.json"
)

for path in [
    TRAIN_DIR,
    VAL_DIR,
    TEST_DIR,
    HAT_MODEL_PATH,
    CNN_MODEL_PATH,
    NORMALIZATION_PATH,
]:
    print(path, "→", path.exists())

    if not path.exists():
        raise FileNotFoundError(path)


Mounted at /content/drive
/content/drive/MyDrive/Super_Resolution_28-07-2026/Wald_Data_GSD/Patches/train → True
/content/drive/MyDrive/Super_Resolution_28-07-2026/Wald_Data_GSD/Patches/val → True
/content/drive/MyDrive/Super_Resolution_28-07-2026/Wald_Data_GSD/Patches/test → True
/content/drive/MyDrive/Super_Resolution_28-07-2026/WV3_Transfer_HAT_Results/best_wv3_transfer_hat_6band.pth → True
/content/drive/MyDrive/Super_Resolution_28-07-2026/Fusion_Baseline_Results/best_fusion_baseline.pth → True
/content/drive/MyDrive/Super_Resolution_28-07-2026/Fusion_Baseline_Results/train_normalization_stats.json → True


## 5) إعدادات التدريب

ابدأ بهذه القيم:

```text
Stage A: تدريب البوابة فقط — 8 Epochs
Stage B: Fine-tuning محدود — 10 Epochs
```

لو حدث نفاد ذاكرة، اترك `BATCH_SIZE=1` كما هو.


In [5]:
import os
import random
import json
import numpy as np

SEED = 42

BATCH_SIZE = 1
ACCUMULATION_STEPS = 4
NUM_WORKERS = 0

STAGE_A_EPOCHS = 8
STAGE_B_EPOCHS = 10

STAGE_A_LR = 1e-4

GATE_STAGE_B_LR = 5e-5
EXPERT_STAGE_B_LR = 5e-6

WEIGHT_DECAY = 1e-6

SCALE = 4
MS_BANDS = 6

# أوزان الـLoss — نقاط بداية للـAblation
LAMBDA_SAM = 0.05
LAMBDA_GRAD = 0.10
LAMBDA_PAN = 0.05
LAMBDA_LR = 0.10
LAMBDA_GATE = 0.20
LAMBDA_BRANCH = 0.05

ORACLE_TEMPERATURE = 0.03
CHARBONNIER_EPS = 1e-3

USE_AMP = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda")

print("Effective batch:", BATCH_SIZE * ACCUMULATION_STEPS)
print("Training precision: float32")


Effective batch: 4
Training precision: float32


## 6) تحميل Normalization

In [6]:
with open(
    NORMALIZATION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stats = json.load(file)

MS_LOW = np.array(
    [
        item["p01"]
        for item in stats["ms"]
    ],
    dtype=np.float32,
)[:, None, None]

MS_HIGH = np.array(
    [
        item["p99"]
        for item in stats["ms"]
    ],
    dtype=np.float32,
)[:, None, None]

PAN_LOW = np.float32(
    stats["pan"]["p01"]
)

PAN_HIGH = np.float32(
    stats["pan"]["p99"]
)


def normalize_ms(array):
    array = array.astype(np.float32)

    return np.clip(
        (array - MS_LOW)
        / (MS_HIGH - MS_LOW),
        0,
        1,
    )


def normalize_pan(array):
    array = array.astype(np.float32)

    return np.clip(
        (array - PAN_LOW)
        / (PAN_HIGH - PAN_LOW),
        0,
        1,
    )

print("Normalization loaded.")


Normalization loaded.


## 7) Dataset وDataLoader

In [7]:
from torch.utils.data import Dataset, DataLoader


class WaldPatchDataset(Dataset):
    def __init__(self, folder, augment=False):
        self.files = sorted(
            Path(folder).glob("*.npz")
        )

        self.augment = augment

        if not self.files:
            raise RuntimeError(
                f"No files in {folder}"
            )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        file_path = self.files[index]

        with np.load(file_path) as sample:
            lr_ms = sample["lr_ms"].copy()
            pan = sample["pan"].copy()
            target = sample["target_ms"].copy()

        lr_ms = torch.from_numpy(
            normalize_ms(lr_ms)
        ).float()

        pan = torch.from_numpy(
            normalize_pan(pan)
        ).float()

        target = torch.from_numpy(
            normalize_ms(target)
        ).float()

        if self.augment:
            if random.random() < 0.5:
                lr_ms = torch.flip(
                    lr_ms,
                    dims=[2],
                )

                pan = torch.flip(
                    pan,
                    dims=[2],
                )

                target = torch.flip(
                    target,
                    dims=[2],
                )

            if random.random() < 0.5:
                lr_ms = torch.flip(
                    lr_ms,
                    dims=[1],
                )

                pan = torch.flip(
                    pan,
                    dims=[1],
                )

                target = torch.flip(
                    target,
                    dims=[1],
                )

            rotations = random.randint(0, 3)

            if rotations:
                lr_ms = torch.rot90(
                    lr_ms,
                    rotations,
                    dims=[1, 2],
                )

                pan = torch.rot90(
                    pan,
                    rotations,
                    dims=[1, 2],
                )

                target = torch.rot90(
                    target,
                    rotations,
                    dims=[1, 2],
                )

        return {
            "lr_ms": lr_ms,
            "pan": pan,
            "target_ms": target,
            "file": file_path.name,
        }


train_dataset = WaldPatchDataset(
    TRAIN_DIR,
    augment=True,
)

val_dataset = WaldPatchDataset(
    VAL_DIR,
    augment=False,
)

test_dataset = WaldPatchDataset(
    TEST_DIR,
    augment=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))


Train: 231
Val  : 22
Test : 22


## 8) تعريف Fusion CNN Expert

In [8]:
import torch.nn as nn
import torch.nn.functional as F


class CNNResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                channels,
                channels,
                3,
                1,
                1,
            ),
            nn.GELU(),
            nn.Conv2d(
                channels,
                channels,
                3,
                1,
                1,
            ),
        )

    def forward(self, x):
        return x + self.block(x)


class FusionCNNExpert(nn.Module):
    def __init__(
        self,
        features=48,
        residual_blocks=6,
    ):
        super().__init__()

        self.head = nn.Conv2d(
            7,
            features,
            3,
            1,
            1,
        )

        self.body = nn.Sequential(
            *[
                CNNResidualBlock(features)
                for _ in range(residual_blocks)
            ]
        )

        self.tail = nn.Sequential(
            nn.Conv2d(
                features,
                features,
                3,
                1,
                1,
            ),
            nn.GELU(),
            nn.Conv2d(
                features,
                6,
                3,
                1,
                1,
            ),
        )

    def forward(self, lr_ms, pan):
        upsampled_ms = F.interpolate(
            lr_ms,
            size=pan.shape[-2:],
            mode="bicubic",
            align_corners=False,
        )

        features = self.head(
            torch.cat(
                [upsampled_ms, pan],
                dim=1,
            )
        )

        features = self.body(features)
        residual = self.tail(features)

        return (
            upsampled_ms + residual
        ).clamp(0, 1)


## 9) تعريف WV3-Transfer HAT Expert

In [9]:
class HATRefinementBlock(nn.Module):
    def __init__(self, channels=64):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                channels,
                channels,
                3,
                1,
                1,
            ),
            nn.GELU(),
            nn.Conv2d(
                channels,
                channels,
                3,
                1,
                1,
            ),
        )

    def forward(self, x):
        return x + self.block(x)


class HATExpert(nn.Module):
    def __init__(self):
        super().__init__()

        self.hat = HAT(
            upscale=4,
            in_chans=7,
            img_size=64,
            window_size=16,
            compress_ratio=3,
            squeeze_factor=30,
            conv_scale=0.01,
            overlap_ratio=0.5,
            img_range=1.0,
            depths=[6, 6, 6, 6, 6, 6],
            embed_dim=180,
            num_heads=[6, 6, 6, 6, 6, 6],
            mlp_ratio=2,
            upsampler="pixelshuffle",
            resi_connection="1conv",
            use_checkpoint=False,
            drop_path_rate=0.0,
        )

        self.hat.conv_last = nn.Conv2d(
            64,
            6,
            3,
            1,
            1,
        )

        self.refine_head = nn.Conv2d(
            13,
            64,
            3,
            1,
            1,
        )

        self.refine_body = nn.Sequential(
            *[
                HATRefinementBlock(64)
                for _ in range(4)
            ]
        )

        self.refine_tail = nn.Conv2d(
            64,
            6,
            3,
            1,
            1,
        )

    def forward(self, lr_ms, pan_hr):
        upsampled_ms = F.interpolate(
            lr_ms,
            size=pan_hr.shape[-2:],
            mode="bicubic",
            align_corners=False,
        )

        pan_lr = F.interpolate(
            pan_hr,
            size=lr_ms.shape[-2:],
            mode="area",
        )

        hat_residual = self.hat(
            torch.cat(
                [lr_ms, pan_lr],
                dim=1,
            )
        )

        coarse_ms = (
            upsampled_ms
            + hat_residual
        )

        features = self.refine_head(
            torch.cat(
                [
                    coarse_ms,
                    upsampled_ms,
                    pan_hr,
                ],
                dim=1,
            )
        )

        features = self.refine_body(features)
        refinement = self.refine_tail(features)

        return (
            coarse_ms + refinement
        ).clamp(0, 1)


## 10) Adaptive Pixel-and-Band Gate

مدخل البوابة:

```text
HAT Prediction       6
CNN Prediction       6
Bicubic MS           6
PAN                   1
|HAT − CNN|          6
-----------------------
Total                25 Channels
```

الناتج:

```text
2 Experts × 6 Bands = 12 weight maps
```


In [10]:
class GateResidualBlock(nn.Module):
    def __init__(self, channels=64):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                channels,
                channels,
                3,
                1,
                1,
            ),
            nn.GELU(),
            nn.Conv2d(
                channels,
                channels,
                3,
                1,
                1,
            ),
        )

    def forward(self, x):
        return x + self.block(x)


class AdaptiveGate(nn.Module):
    def __init__(self):
        super().__init__()

        self.head = nn.Sequential(
            nn.Conv2d(
                25,
                64,
                3,
                1,
                1,
            ),
            nn.GELU(),
        )

        self.body = nn.Sequential(
            GateResidualBlock(64),
            GateResidualBlock(64),
            GateResidualBlock(64),
        )

        self.tail = nn.Conv2d(
            64,
            12,
            1,
            1,
            0,
        )

        # يبدأ قريبًا من متوسط 50/50
        nn.init.zeros_(
            self.tail.weight
        )

        nn.init.zeros_(
            self.tail.bias
        )

    def forward(
        self,
        hat_prediction,
        cnn_prediction,
        bicubic,
        pan,
    ):
        disagreement = torch.abs(
            hat_prediction
            - cnn_prediction
        )

        gate_input = torch.cat(
            [
                hat_prediction,
                cnn_prediction,
                bicubic,
                pan,
                disagreement,
            ],
            dim=1,
        )

        features = self.head(gate_input)
        features = self.body(features)
        logits = self.tail(features)

        batch, _, height, width = logits.shape

        logits = logits.view(
            batch,
            2,
            6,
            height,
            width,
        )

        weights = torch.softmax(
            logits,
            dim=1,
        )

        return weights


## 11) النموذج الهجين الكامل

In [11]:
class AdaptiveHATCNNFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.hat_expert = HATExpert()
        self.cnn_expert = FusionCNNExpert()
        self.gate = AdaptiveGate()

    def forward(
        self,
        lr_ms,
        pan,
        detach_experts=False,
    ):
        bicubic = F.interpolate(
            lr_ms,
            size=pan.shape[-2:],
            mode="bicubic",
            align_corners=False,
        ).clamp(0, 1)

        if detach_experts:
            with torch.no_grad():
                hat_prediction = self.hat_expert(
                    lr_ms,
                    pan,
                )

                cnn_prediction = self.cnn_expert(
                    lr_ms,
                    pan,
                )

        else:
            hat_prediction = self.hat_expert(
                lr_ms,
                pan,
            )

            cnn_prediction = self.cnn_expert(
                lr_ms,
                pan,
            )

        weights = self.gate(
            hat_prediction,
            cnn_prediction,
            bicubic,
            pan,
        )

        hat_residual = (
            hat_prediction
            - bicubic
        )

        cnn_residual = (
            cnn_prediction
            - bicubic
        )

        final_prediction = (
            bicubic
            + weights[:, 0] * hat_residual
            + weights[:, 1] * cnn_residual
        ).clamp(0, 1)

        return {
            "final": final_prediction,
            "hat": hat_prediction,
            "cnn": cnn_prediction,
            "bicubic": bicubic,
            "weights": weights,
        }


model = AdaptiveHATCNNFusion().to(device)

print(
    "Total parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Total parameters: 21,599,830


## 12) تحميل أوزان الخبيرين

In [12]:
hat_checkpoint = torch.load(
    HAT_MODEL_PATH,
    map_location="cpu",
    weights_only=False,
)

cnn_checkpoint = torch.load(
    CNN_MODEL_PATH,
    map_location="cpu",
    weights_only=False,
)

model.hat_expert.load_state_dict(
    hat_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

model.cnn_expert.load_state_dict(
    cnn_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

print("HAT expert loaded.")
print("CNN expert loaded.")


HAT expert loaded.
CNN expert loaded.


## 13) Numerical Safety Check

In [13]:
debug_batch = next(
    iter(train_loader)
)

debug_lr = debug_batch[
    "lr_ms"
][:1].to(device)

debug_pan = debug_batch[
    "pan"
][:1].to(device)

debug_target = debug_batch[
    "target_ms"
][:1].to(device)

model.eval()

with torch.inference_mode():
    debug_outputs = model(
        debug_lr,
        debug_pan,
        detach_experts=True,
    )

for name in [
    "final",
    "hat",
    "cnn",
    "bicubic",
    "weights",
]:
    tensor = debug_outputs[name]

    print(
        name,
        tensor.shape,
        "finite:",
        torch.isfinite(
            tensor
        ).all().item(),
    )

    assert torch.isfinite(
        tensor
    ).all()

print(
    "Mean HAT weight:",
    float(
        debug_outputs[
            "weights"
        ][:, 0].mean()
    ),
)

print(
    "Mean CNN weight:",
    float(
        debug_outputs[
            "weights"
        ][:, 1].mean()
    ),
)


final torch.Size([1, 6, 256, 256]) finite: True
hat torch.Size([1, 6, 256, 256]) finite: True
cnn torch.Size([1, 6, 256, 256]) finite: True
bicubic torch.Size([1, 6, 256, 256]) finite: True
weights torch.Size([1, 2, 6, 256, 256]) finite: True
Mean HAT weight: 0.5
Mean CNN weight: 0.5


## 14) تعريف Loss Functions

In [14]:
import math


def charbonnier_loss(
    prediction,
    target,
    epsilon=1e-3,
):
    difference = (
        prediction - target
    )

    return torch.mean(
        torch.sqrt(
            difference * difference
            + epsilon * epsilon
        )
    )


def spectral_angle_cosine_loss(
    prediction,
    target,
):
    dot = torch.sum(
        prediction * target,
        dim=1,
    )

    prediction_norm = (
        torch.linalg.vector_norm(
            prediction,
            dim=1,
        )
    )

    target_norm = (
        torch.linalg.vector_norm(
            target,
            dim=1,
        )
    )

    cosine = dot / torch.clamp(
        prediction_norm
        * target_norm,
        min=1e-8,
    )

    cosine = torch.clamp(
        cosine,
        -1,
        1,
    )

    return (
        1.0 - cosine
    ).mean()


def spatial_gradients(image):
    gradient_x = (
        image[:, :, :, 1:]
        - image[:, :, :, :-1]
    )

    gradient_y = (
        image[:, :, 1:, :]
        - image[:, :, :-1, :]
    )

    return gradient_x, gradient_y


def gradient_loss(
    prediction,
    target,
):
    prediction_x, prediction_y = (
        spatial_gradients(
            prediction
        )
    )

    target_x, target_y = (
        spatial_gradients(
            target
        )
    )

    return (
        F.l1_loss(
            prediction_x,
            target_x,
        )
        + F.l1_loss(
            prediction_y,
            target_y,
        )
    )


def high_pass(image, kernel_size=5):
    low_pass = F.avg_pool2d(
        image,
        kernel_size=kernel_size,
        stride=1,
        padding=kernel_size // 2,
    )

    return image - low_pass


def negative_correlation_loss(
    first,
    second,
):
    first = first.flatten(
        start_dim=1
    )

    second = second.flatten(
        start_dim=1
    )

    first = (
        first
        - first.mean(
            dim=1,
            keepdim=True,
        )
    )

    second = (
        second
        - second.mean(
            dim=1,
            keepdim=True,
        )
    )

    numerator = torch.sum(
        first * second,
        dim=1,
    )

    denominator = torch.sqrt(
        torch.sum(
            first * first,
            dim=1,
        )
        * torch.sum(
            second * second,
            dim=1,
        )
        + 1e-8
    )

    correlation = (
        numerator
        / denominator
    )

    return (
        1.0 - correlation
    ).mean()


def pan_highpass_loss(
    prediction,
    pan,
):
    # Mean intensity is an approximation because
    # the exact sensor spectral response is unavailable.
    prediction_intensity = (
        prediction.mean(
            dim=1,
            keepdim=True,
        )
    )

    return negative_correlation_loss(
        high_pass(
            prediction_intensity
        ),
        high_pass(pan),
    )


def low_resolution_consistency_loss(
    prediction,
    lr_ms,
):
    downsampled = F.interpolate(
        prediction,
        size=lr_ms.shape[-2:],
        mode="area",
    )

    return F.l1_loss(
        downsampled,
        lr_ms,
    )


def oracle_gate_targets(
    hat_prediction,
    cnn_prediction,
    target,
    temperature=0.03,
):
    hat_error = torch.abs(
        hat_prediction - target
    )

    cnn_error = torch.abs(
        cnn_prediction - target
    )

    errors = torch.stack(
        [
            hat_error,
            cnn_error,
        ],
        dim=1,
    )

    return torch.softmax(
        -errors / temperature,
        dim=1,
    ).detach()


def oracle_gate_loss(
    predicted_weights,
    oracle_weights,
):
    return (
        -oracle_weights
        * torch.log(
            predicted_weights.clamp_min(
                1e-8
            )
        )
    ).sum(
        dim=1
    ).mean()


def branch_stabilization_loss(
    hat_prediction,
    cnn_prediction,
    target,
):
    return 0.5 * (
        charbonnier_loss(
            hat_prediction,
            target,
            CHARBONNIER_EPS,
        )
        + charbonnier_loss(
            cnn_prediction,
            target,
            CHARBONNIER_EPS,
        )
    )


def total_fusion_loss(
    outputs,
    target,
    lr_ms,
    pan,
    include_branch_loss,
):
    final_prediction = outputs[
        "final"
    ]

    reconstruction = charbonnier_loss(
        final_prediction,
        target,
        CHARBONNIER_EPS,
    )

    sam = spectral_angle_cosine_loss(
        final_prediction,
        target,
    )

    gradient = gradient_loss(
        final_prediction,
        target,
    )

    pan_detail = pan_highpass_loss(
        final_prediction,
        pan,
    )

    lr_consistency = (
        low_resolution_consistency_loss(
            final_prediction,
            lr_ms,
        )
    )

    oracle_weights = (
        oracle_gate_targets(
            outputs["hat"],
            outputs["cnn"],
            target,
            ORACLE_TEMPERATURE,
        )
    )

    gate_supervision = oracle_gate_loss(
        outputs["weights"],
        oracle_weights,
    )

    branch_loss = torch.zeros(
        (),
        device=target.device,
    )

    if include_branch_loss:
        branch_loss = (
            branch_stabilization_loss(
                outputs["hat"],
                outputs["cnn"],
                target,
            )
        )

    total = (
        reconstruction
        + LAMBDA_SAM * sam
        + LAMBDA_GRAD * gradient
        + LAMBDA_PAN * pan_detail
        + LAMBDA_LR * lr_consistency
        + LAMBDA_GATE * gate_supervision
        + LAMBDA_BRANCH * branch_loss
    )

    components = {
        "total": total,
        "reconstruction": reconstruction,
        "sam_loss": sam,
        "gradient_loss": gradient,
        "pan_loss": pan_detail,
        "lr_consistency": lr_consistency,
        "gate_loss": gate_supervision,
        "branch_loss": branch_loss,
    }

    return total, components


## 15) Metrics

In [15]:
def batch_psnr(
    prediction,
    target,
):
    mse = torch.mean(
        (prediction - target) ** 2,
        dim=(1, 2, 3),
    )

    return (
        10
        * torch.log10(
            1.0
            / torch.clamp(
                mse,
                min=1e-12,
            )
        )
    ).mean()


def batch_sam_degrees(
    prediction,
    target,
):
    ms_low = torch.from_numpy(
        MS_LOW
    ).to(prediction.device)

    ms_high = torch.from_numpy(
        MS_HIGH
    ).to(prediction.device)

    prediction_dn = (
        prediction
        * (ms_high - ms_low)
        + ms_low
    )

    target_dn = (
        target
        * (ms_high - ms_low)
        + ms_low
    )

    dot = torch.sum(
        prediction_dn
        * target_dn,
        dim=1,
    )

    prediction_norm = (
        torch.linalg.vector_norm(
            prediction_dn,
            dim=1,
        )
    )

    target_norm = (
        torch.linalg.vector_norm(
            target_dn,
            dim=1,
        )
    )

    cosine = dot / torch.clamp(
        prediction_norm
        * target_norm,
        min=1e-8,
    )

    cosine = torch.clamp(
        cosine,
        -1 + 1e-7,
        1 - 1e-7,
    )

    return (
        torch.acos(cosine)
        * (180.0 / math.pi)
    ).mean()


def batch_ergas(
    prediction,
    target,
    scale=4,
):
    rmse = torch.sqrt(
        torch.mean(
            (prediction - target) ** 2,
            dim=(2, 3),
        )
    )

    mean_target = torch.mean(
        target,
        dim=(2, 3),
    ).abs().clamp_min(1e-6)

    relative_squared = (
        rmse / mean_target
    ) ** 2

    ergas = (
        100.0 / scale
        * torch.sqrt(
            relative_squared.mean(
                dim=1
            )
        )
    )

    return ergas.mean()


## 16) تقييم الخبراء والـSimple Average قبل التدريب

In [16]:
from tqdm.auto import tqdm


@torch.inference_mode()
def evaluate_fixed_methods(loader):
    model.eval()

    results = {
        "bicubic": {
            "psnr": [],
            "sam": [],
            "ergas": [],
        },
        "cnn": {
            "psnr": [],
            "sam": [],
            "ergas": [],
        },
        "hat": {
            "psnr": [],
            "sam": [],
            "ergas": [],
        },
        "simple_average": {
            "psnr": [],
            "sam": [],
            "ergas": [],
        },
    }

    for batch in tqdm(
        loader,
        desc="Fixed methods",
    ):
        lr_ms = batch[
            "lr_ms"
        ].to(device)

        pan = batch[
            "pan"
        ].to(device)

        target = batch[
            "target_ms"
        ].to(device)

        outputs = model(
            lr_ms,
            pan,
            detach_experts=True,
        )

        predictions = {
            "bicubic": outputs[
                "bicubic"
            ],
            "cnn": outputs["cnn"],
            "hat": outputs["hat"],
            "simple_average": (
                0.5 * outputs["hat"]
                + 0.5 * outputs["cnn"]
            ).clamp(0, 1),
        }

        for name, prediction in predictions.items():
            results[name][
                "psnr"
            ].append(
                batch_psnr(
                    prediction,
                    target,
                ).item()
            )

            results[name][
                "sam"
            ].append(
                batch_sam_degrees(
                    prediction,
                    target,
                ).item()
            )

            results[name][
                "ergas"
            ].append(
                batch_ergas(
                    prediction,
                    target,
                    SCALE,
                ).item()
            )

    summary = {}

    for name, values in results.items():
        summary[name] = {
            "psnr_db": float(
                np.mean(
                    values["psnr"]
                )
            ),
            "sam_degrees": float(
                np.mean(
                    values["sam"]
                )
            ),
            "ergas": float(
                np.mean(
                    values["ergas"]
                )
            ),
        }

    return summary


fixed_test_metrics = evaluate_fixed_methods(
    test_loader
)

print(
    json.dumps(
        fixed_test_metrics,
        indent=2,
        ensure_ascii=False,
    )
)


Fixed methods:   0%|          | 0/22 [00:00<?, ?it/s]

{
  "bicubic": {
    "psnr_db": 18.68678769198331,
    "sam_degrees": 2.6411342945965854,
    "ergas": 8.514944141561335
  },
  "cnn": {
    "psnr_db": 25.076181931929156,
    "sam_degrees": 1.9011461572213606,
    "ergas": 4.1111755262721665
  },
  "hat": {
    "psnr_db": 25.35746296969327,
    "sam_degrees": 1.910725479776209,
    "ergas": 3.9757342663678257
  },
  "simple_average": {
    "psnr_db": 25.290936036543414,
    "sam_degrees": 1.8842810284007678,
    "ergas": 4.008538441224531
  }
}


## 17) إعداد مراحل التدريب

In [17]:
def freeze_all(module):
    for parameter in module.parameters():
        parameter.requires_grad = False


def configure_stage_a(model):
    freeze_all(
        model.hat_expert
    )

    freeze_all(
        model.cnn_expert
    )

    for parameter in model.gate.parameters():
        parameter.requires_grad = True


def configure_stage_b(model):
    freeze_all(
        model.hat_expert
    )

    freeze_all(
        model.cnn_expert
    )

    # HAT: آخر مجموعتين وطبقات إعادة البناء
    for layer in model.hat_expert.hat.layers[-2:]:
        for parameter in layer.parameters():
            parameter.requires_grad = True

    for module in [
        model.hat_expert.hat.norm,
        model.hat_expert.hat.conv_after_body,
        model.hat_expert.hat.conv_before_upsample,
        model.hat_expert.hat.upsample,
        model.hat_expert.hat.conv_last,
        model.hat_expert.refine_head,
        model.hat_expert.refine_body,
        model.hat_expert.refine_tail,
    ]:
        for parameter in module.parameters():
            parameter.requires_grad = True

    # CNN: آخر Residual Blocks والـTail
    for block in model.cnn_expert.body[-2:]:
        for parameter in block.parameters():
            parameter.requires_grad = True

    for parameter in model.cnn_expert.tail.parameters():
        parameter.requires_grad = True

    for parameter in model.gate.parameters():
        parameter.requires_grad = True


def count_trainable(module):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if parameter.requires_grad
    )


## 18) Training and Evaluation Functions

In [18]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau


def create_stage_a_optimizer():
    return AdamW(
        model.gate.parameters(),
        lr=STAGE_A_LR,
        weight_decay=WEIGHT_DECAY,
    )


def create_stage_b_optimizer():
    gate_parameters = [
        parameter
        for parameter in model.gate.parameters()
        if parameter.requires_grad
    ]

    expert_parameters = [
        parameter
        for name, parameter in model.named_parameters()
        if (
            parameter.requires_grad
            and not name.startswith("gate.")
        )
    ]

    return AdamW(
        [
            {
                "params": gate_parameters,
                "lr": GATE_STAGE_B_LR,
            },
            {
                "params": expert_parameters,
                "lr": EXPERT_STAGE_B_LR,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )


def train_one_epoch(
    loader,
    optimizer,
    detach_experts,
    include_branch_loss,
):
    model.train()

    if detach_experts:
        model.hat_expert.eval()
        model.cnn_expert.eval()

    optimizer.zero_grad(
        set_to_none=True
    )

    totals = {
        "loss": 0.0,
        "psnr": 0.0,
        "sam": 0.0,
        "samples": 0,
    }

    progress = tqdm(
        loader,
        desc="Train",
        leave=False,
    )

    for batch_index, batch in enumerate(
        progress,
        start=1,
    ):
        lr_ms = batch[
            "lr_ms"
        ].to(
            device,
            non_blocking=True,
        )

        pan = batch[
            "pan"
        ].to(
            device,
            non_blocking=True,
        )

        target = batch[
            "target_ms"
        ].to(
            device,
            non_blocking=True,
        )

        outputs = model(
            lr_ms,
            pan,
            detach_experts=detach_experts,
        )

        if not torch.isfinite(
            outputs["final"]
        ).all():
            raise FloatingPointError(
                f"Non-finite prediction at batch {batch_index}"
            )

        loss, components = total_fusion_loss(
            outputs,
            target,
            lr_ms,
            pan,
            include_branch_loss,
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite loss at batch {batch_index}"
            )

        (
            loss
            / ACCUMULATION_STEPS
        ).backward()

        should_step = (
            batch_index
            % ACCUMULATION_STEPS
            == 0
            or batch_index
            == len(loader)
        )

        if should_step:
            for name, parameter in model.named_parameters():
                if (
                    parameter.requires_grad
                    and parameter.grad is not None
                    and not torch.isfinite(
                        parameter.grad
                    ).all()
                ):
                    raise FloatingPointError(
                        f"Non-finite gradient in {name}"
                    )

            torch.nn.utils.clip_grad_norm_(
                [
                    parameter
                    for parameter in model.parameters()
                    if parameter.requires_grad
                ],
                max_norm=1.0,
            )

            optimizer.step()

            optimizer.zero_grad(
                set_to_none=True
            )

        batch_size = lr_ms.shape[0]

        totals["loss"] += (
            loss.item()
            * batch_size
        )

        totals["psnr"] += (
            batch_psnr(
                outputs["final"].detach(),
                target,
            ).item()
            * batch_size
        )

        totals["sam"] += (
            batch_sam_degrees(
                outputs["final"].detach(),
                target,
            ).item()
            * batch_size
        )

        totals["samples"] += batch_size

        progress.set_postfix(
            total=f"{loss.item():.4f}",
            recon=f"{components['reconstruction'].item():.4f}",
            gate=f"{components['gate_loss'].item():.4f}",
        )

    return {
        "loss": (
            totals["loss"]
            / totals["samples"]
        ),
        "psnr_db": (
            totals["psnr"]
            / totals["samples"]
        ),
        "sam_degrees": (
            totals["sam"]
            / totals["samples"]
        ),
    }


@torch.inference_mode()
def evaluate_adaptive(loader):
    model.eval()

    totals = {
        "loss": 0.0,
        "psnr": 0.0,
        "sam": 0.0,
        "ergas": 0.0,
        "hat_weight": 0.0,
        "cnn_weight": 0.0,
        "samples": 0,
    }

    for batch in tqdm(
        loader,
        desc="Evaluation",
        leave=False,
    ):
        lr_ms = batch[
            "lr_ms"
        ].to(device)

        pan = batch[
            "pan"
        ].to(device)

        target = batch[
            "target_ms"
        ].to(device)

        outputs = model(
            lr_ms,
            pan,
            detach_experts=False,
        )

        loss, _ = total_fusion_loss(
            outputs,
            target,
            lr_ms,
            pan,
            include_branch_loss=False,
        )

        batch_size = lr_ms.shape[0]

        totals["loss"] += (
            loss.item()
            * batch_size
        )

        totals["psnr"] += (
            batch_psnr(
                outputs["final"],
                target,
            ).item()
            * batch_size
        )

        totals["sam"] += (
            batch_sam_degrees(
                outputs["final"],
                target,
            ).item()
            * batch_size
        )

        totals["ergas"] += (
            batch_ergas(
                outputs["final"],
                target,
                SCALE,
            ).item()
            * batch_size
        )

        totals["hat_weight"] += (
            outputs["weights"][
                :, 0
            ].mean().item()
            * batch_size
        )

        totals["cnn_weight"] += (
            outputs["weights"][
                :, 1
            ].mean().item()
            * batch_size
        )

        totals["samples"] += batch_size

    return {
        "loss": (
            totals["loss"]
            / totals["samples"]
        ),
        "psnr_db": (
            totals["psnr"]
            / totals["samples"]
        ),
        "sam_degrees": (
            totals["sam"]
            / totals["samples"]
        ),
        "ergas": (
            totals["ergas"]
            / totals["samples"]
        ),
        "mean_hat_weight": (
            totals["hat_weight"]
            / totals["samples"]
        ),
        "mean_cnn_weight": (
            totals["cnn_weight"]
            / totals["samples"]
        ),
    }


def train_stage(
    stage_name,
    epochs,
    optimizer,
    detach_experts,
    include_branch_loss,
    history,
    best_val_loss,
    best_path,
):
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
    )

    print("\n" + "=" * 72)
    print(stage_name)
    print(
        "Trainable parameters:",
        f"{count_trainable(model):,}",
    )
    print("=" * 72)

    for epoch in range(
        1,
        epochs + 1,
    ):
        train_metrics = train_one_epoch(
            train_loader,
            optimizer,
            detach_experts,
            include_branch_loss,
        )

        val_metrics = evaluate_adaptive(
            val_loader
        )

        scheduler.step(
            val_metrics["loss"]
        )

        record = {
            "stage": stage_name,
            "epoch": epoch,
            "train": train_metrics,
            "validation": val_metrics,
            "learning_rates": [
                group["lr"]
                for group in optimizer.param_groups
            ],
        }

        history.append(record)

        print(
            f"{stage_name} | "
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss {train_metrics['loss']:.5f} | "
            f"Val Loss {val_metrics['loss']:.5f} | "
            f"Val PSNR {val_metrics['psnr_db']:.3f} dB | "
            f"Val SAM {val_metrics['sam_degrees']:.3f}° | "
            f"HAT weight {val_metrics['mean_hat_weight']:.3f}"
        )

        checkpoint = {
            "model_state_dict": model.state_dict(),
            "stage": stage_name,
            "epoch": epoch,
            "validation": val_metrics,
            "loss_weights": {
                "sam": LAMBDA_SAM,
                "gradient": LAMBDA_GRAD,
                "pan": LAMBDA_PAN,
                "low_resolution": LAMBDA_LR,
                "oracle_gate": LAMBDA_GATE,
                "branch": LAMBDA_BRANCH,
            },
            "architecture": {
                "experts": [
                    "WV3-Pretrained HAT-PAN",
                    "Fusion CNN",
                ],
                "gate": "Pixel-and-band adaptive softmax",
                "gate_input_channels": 25,
                "gate_output_maps": 12,
            },
        }

        torch.save(
            checkpoint,
            LAST_MODEL_PATH,
        )

        if (
            val_metrics["loss"]
            < best_val_loss
        ):
            best_val_loss = (
                val_metrics["loss"]
            )

            torch.save(
                checkpoint,
                best_path,
            )

            print(
                "Saved new best model."
            )

    return history, best_val_loss


## 19) Stage A — تدريب البوابة فقط

In [19]:
for path in [
    STAGE_A_PATH,
    BEST_MODEL_PATH,
    LAST_MODEL_PATH,
    HISTORY_PATH,
    TEST_METRICS_PATH,
]:
    if path.exists():
        path.unlink()

        print(
            "Deleted stale file:",
            path,
        )

configure_stage_a(model)

history = []
best_val_loss = float("inf")

stage_a_optimizer = (
    create_stage_a_optimizer()
)

history, best_val_loss = train_stage(
    stage_name="Stage A — Gate Only",
    epochs=STAGE_A_EPOCHS,
    optimizer=stage_a_optimizer,
    detach_experts=True,
    include_branch_loss=False,
    history=history,
    best_val_loss=best_val_loss,
    best_path=STAGE_A_PATH,
)



Stage A — Gate Only
Trainable parameters: 236,812


Train:   0%|          | 0/231 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/22 [00:00<?, ?it/s]

Stage A — Gate Only | Epoch 01/8 | Train Loss 0.18300 | Val Loss 0.18109 | Val PSNR 26.011 dB | Val SAM 1.706° | HAT weight 0.508
Saved new best model.


Train:   0%|          | 0/231 [00:00<?, ?it/s]

...

Train:   0%|          | 0/231 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/22 [00:00<?, ?it/s]

Stage A — Gate Only | Epoch 05/8 | Train Loss 0.18272 | Val Loss 0.18082 | Val PSNR 26.037 dB | Val SAM 1.705° | HAT weight 0.514
Saved new best model.


Train:   0%|          | 0/231 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 20) Stage B — Fine-Tuning محدود للخبيرين والبوابة

In [ ]:
stage_a_checkpoint = torch.load(
    STAGE_A_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    stage_a_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

configure_stage_b(model)

stage_b_optimizer = (
    create_stage_b_optimizer()
)

history, best_val_loss = train_stage(
    stage_name="Stage B — Joint Fine-Tuning",
    epochs=STAGE_B_EPOCHS,
    optimizer=stage_b_optimizer,
    detach_experts=False,
    include_branch_loss=True,
    history=history,
    best_val_loss=best_val_loss,
    best_path=BEST_MODEL_PATH,
)

with open(
    HISTORY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        history,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    "Best adaptive model:",
    BEST_MODEL_PATH,
)


## 21) منحنيات التدريب

In [ ]:
import matplotlib.pyplot as plt

epoch_axis = list(
    range(
        1,
        len(history) + 1,
    )
)

train_loss = [
    item["train"]["loss"]
    for item in history
]

val_loss = [
    item["validation"]["loss"]
    for item in history
]

val_psnr = [
    item["validation"]["psnr_db"]
    for item in history
]

val_sam = [
    item["validation"]["sam_degrees"]
    for item in history
]

hat_weights = [
    item["validation"]["mean_hat_weight"]
    for item in history
]


plt.figure(figsize=(10, 6))

plt.plot(
    epoch_axis,
    train_loss,
    label="Train total loss",
)

plt.plot(
    epoch_axis,
    val_loss,
    label="Validation total loss",
)

plt.axvline(
    STAGE_A_EPOCHS,
    linestyle="--",
    label="Stage B starts",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(
    "Adaptive HAT-CNN Fusion Loss"
)
plt.legend()
plt.grid()
plt.show()


plt.figure(figsize=(10, 6))

plt.plot(
    epoch_axis,
    val_psnr,
    label="Validation PSNR",
)

plt.xlabel("Epoch")
plt.ylabel("PSNR (dB)")
plt.title(
    "Adaptive Fusion Validation PSNR"
)
plt.grid()
plt.show()


plt.figure(figsize=(10, 6))

plt.plot(
    epoch_axis,
    val_sam,
)

plt.xlabel("Epoch")
plt.ylabel("SAM (degrees)")
plt.title(
    "Adaptive Fusion Validation SAM"
)
plt.grid()
plt.show()


plt.figure(figsize=(10, 6))

plt.plot(
    epoch_axis,
    hat_weights,
    label="Mean HAT weight",
)

plt.axhline(
    0.5,
    linestyle="--",
    label="Equal weighting",
)

plt.xlabel("Epoch")
plt.ylabel("Mean gate weight")
plt.title(
    "Learned HAT Contribution"
)
plt.legend()
plt.grid()
plt.show()


## 22) Test والمقارنة النهائية

In [ ]:
best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

adaptive_test_metrics = (
    evaluate_adaptive(
        test_loader
    )
)

comparison = {
    "fixed_methods_recomputed": (
        fixed_test_metrics
    ),
    "adaptive_hat_cnn_fusion": (
        adaptive_test_metrics
    ),
}

if CNN_METRICS_PATH.exists():
    with open(
        CNN_METRICS_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        comparison[
            "previous_cnn_metrics"
        ] = json.load(file)

if HAT_METRICS_PATH.exists():
    with open(
        HAT_METRICS_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        comparison[
            "previous_hat_metrics"
        ] = json.load(file)

comparison[
    "adaptive_vs_hat_psnr_db"
] = (
    adaptive_test_metrics["psnr_db"]
    - fixed_test_metrics["hat"]["psnr_db"]
)

comparison[
    "adaptive_vs_cnn_psnr_db"
] = (
    adaptive_test_metrics["psnr_db"]
    - fixed_test_metrics["cnn"]["psnr_db"]
)

comparison[
    "adaptive_vs_hat_sam_degrees"
] = (
    fixed_test_metrics["hat"]["sam_degrees"]
    - adaptive_test_metrics["sam_degrees"]
)

comparison[
    "adaptive_vs_cnn_sam_degrees"
] = (
    fixed_test_metrics["cnn"]["sam_degrees"]
    - adaptive_test_metrics["sam_degrees"]
)

with open(
    TEST_METRICS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        comparison,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    json.dumps(
        comparison,
        indent=2,
        ensure_ascii=False,
    )
)

print(
    "Saved:",
    TEST_METRICS_PATH,
)


## 23) عرض النتائج وخرائط البوابة

In [ ]:
test_batch = next(
    iter(test_loader)
)

lr_ms = test_batch[
    "lr_ms"
].to(device)

pan = test_batch[
    "pan"
].to(device)

target = test_batch[
    "target_ms"
].to(device)

model.eval()

with torch.inference_mode():
    outputs = model(
        lr_ms,
        pan,
        detach_experts=False,
    )


def rgb321_tensor(tensor):
    return torch.stack(
        [
            tensor[2],
            tensor[1],
            tensor[0],
        ],
        dim=-1,
    ).float().clamp(
        0,
        1,
    ).cpu().numpy()


for title, tensor in [
    ("Bicubic", outputs["bicubic"][0]),
    ("Fusion CNN Expert", outputs["cnn"][0]),
    ("WV3-Transfer HAT Expert", outputs["hat"][0]),
    ("Adaptive HAT-CNN Fusion", outputs["final"][0]),
    ("Ground Truth", target[0]),
]:
    plt.figure(figsize=(8, 8))
    plt.imshow(
        rgb321_tensor(tensor)
    )
    plt.title(title)
    plt.axis("off")
    plt.show()


mean_hat_map = outputs[
    "weights"
][0, 0].mean(
    dim=0
).cpu().numpy()

mean_cnn_map = outputs[
    "weights"
][0, 1].mean(
    dim=0
).cpu().numpy()


plt.figure(figsize=(8, 8))
plt.imshow(
    mean_hat_map,
    vmin=0,
    vmax=1,
)
plt.title(
    "Mean HAT Gate Weight"
)
plt.colorbar()
plt.axis("off")
plt.show()


plt.figure(figsize=(8, 8))
plt.imshow(
    mean_cnn_map,
    vmin=0,
    vmax=1,
)
plt.title(
    "Mean CNN Gate Weight"
)
plt.colorbar()
plt.axis("off")
plt.show()


band_hat_weights = outputs[
    "weights"
][0, 0].mean(
    dim=(1, 2)
).cpu().numpy()

band_cnn_weights = outputs[
    "weights"
][0, 1].mean(
    dim=(1, 2)
).cpu().numpy()

print("Average weights by band:")

for band_index in range(6):
    print(
        f"Band {band_index + 1}: "
        f"HAT={band_hat_weights[band_index]:.4f}, "
        f"CNN={band_cnn_weights[band_index]:.4f}"
    )

print(
    "Prediction finite:",
    torch.isfinite(
        outputs["final"]
    ).all().item(),
)

print(
    "Test file:",
    test_batch["file"][0],
)


# المطلوب بعد التشغيل

أرسل:

1. نتيجة `fixed_test_metrics` قبل التدريب.
2. نتائج Stage A.
3. نتائج Stage B.
4. JSON المقارنة النهائية.
5. صور:
   - Adaptive HAT-CNN Fusion.
   - Ground Truth.
   - Mean HAT Gate Weight.
   - Average weights by band.

## كيف نعرف أن الدمج نجح؟

الهدف المثالي:

```text
PSNR > 25.357 dB
SAM ≤ 1.901°
ERGAS أقل من كل فرع
```

لكن حتى لو تحسن مقياس واحد فقط، نراجع الـGate maps والـAblation قبل الحكم.

## Ablation المطلوبة لاحقًا

بعد نجاح النموذج الأساسي نجرب:

```text
1. Simple Average
2. Adaptive Gate بدون Oracle Loss
3. Adaptive Gate مع Oracle Loss
4. Full multi-loss model
```

وبذلك نعرف بالضبط أي إضافة سببت التحسن.
